# Boltzmann Machines

## Importing the libraries

In [1]:
import torch
import numpy as np
import pandas as pd

## Importing the dataset

In [2]:
movies = pd.read_csv('ml-1m/movies.dat', sep = '::', header = None, engine = 'python', encoding = 'latin-1')
users = pd.read_csv('ml-1m/users.dat', sep = '::', header = None, engine = 'python', encoding = 'latin-1')
ratings = pd.read_csv('ml-1m/ratings.dat', sep = '::', header = None, engine = 'python', encoding = 'latin-1')

## Preparing the training set and the test set

In [3]:
training_set = pd.read_csv('ml-100k/u1.base', delimiter = '\t')
training_set = np.array(training_set, dtype = 'int')

test_set = pd.read_csv('ml-100k/u1.test', delimiter = '\t')
test_set = np.array(test_set, dtype = 'int')

## Getting the number of users and movies

In [4]:
nb_users = int(max(max(training_set[:, 0]), max(test_set[:, 0])))
nb_movies = int(max(max(training_set[:, 1]), max(test_set[:, 1])))

## Converting the data into an array with users in lines and movies in columns

In [5]:
def convert(data):
    new_data = []

    for id_users in range(1, nb_users + 1):

        id_movies = data[:, 1][data[:, 0] == id_users]
        id_ratings = data[:, 2][data[:, 0] == id_users]

        ratings = np.zeros(nb_movies)
        ratings[id_movies - 1] = id_ratings

        new_data.append(list(ratings))

    return new_data

In [6]:
training_set = convert(training_set)
test_set = convert(test_set)

## Converting the data into Torch tensors

In [7]:
training_set = torch.FloatTensor(training_set)
test_set = torch.FloatTensor(test_set)

## Converting the ratings into binary ratings 1 (Liked) or 0 (Not Liked)

In [8]:
training_set[training_set == 0] = -1
training_set[training_set == 1] = 0
training_set[training_set == 2] = 0
training_set[training_set >= 3] = 1

In [9]:
test_set[test_set == 0] = -1
test_set[test_set == 1] = 0
test_set[test_set == 2] = 0
test_set[test_set >= 3] = 1

## Creating the architecture of the Neural Network

In [10]:
class RBM():

    def __init__(self, nv, nh = 100, k = 10):
        self.nv = nv
        self.nh = nh
        self.k = k

        self.W = torch.randn(nh, nv) * 0.1
        self.b = torch.zeros(1, nv)
        self.c = torch.zeros(1, nh)

    def sample_h(self, v):
        wv = torch.mm(v, self.W.t())
        pre_activation = wv + self.c.expand_as(wv)
        p_h_given_v = torch.sigmoid(pre_activation)
        return p_h_given_v, torch.bernoulli(p_h_given_v)

    def sample_v(self, h):
        wh = torch.mm(h, self.W)
        pre_activation = wh + self.b.expand_as(wh)
        p_v_given_h = torch.sigmoid(pre_activation)
        return p_v_given_h, torch.bernoulli(p_v_given_h)

    def train(self, v0, vk, ph0, phk):
        self.W += torch.mm(v0.t(), ph0).t() - torch.mm(vk.t(), phk).t()
        self.b += torch.sum((v0 - vk), 0)
        self.c += torch.sum((ph0 - phk), 0)

    def fit(self, train_data, epochs = 10, batch_size = 100):
        nb_users = train_data.shape[0]
        history = []

        for epoch in range(epochs):
            loss = 0
            counter = 0

            for i in range(0, nb_users - batch_size, batch_size):
                v0 = train_data[i : i + batch_size]
                vk = train_data[i : i + batch_size]
                ph0, _ = self.sample_h(v0)

                for _ in range(self.k):
                    _, hk = self.sample_h(vk)
                    _, vk = self.sample_v(hk)
                    vk[v0 < 0] = v0[v0 < 0]

                phk, _ = self.sample_h(vk)
                self.train(v0, vk, ph0, phk)
                batch_loss = torch.mean(torch.abs(v0[v0 >= 0] - vk[v0 >= 0]))

                loss += batch_loss.item()
                counter += 1

            epoch_loss = loss/counter
            history.append(epoch_loss)

            print(f"Epoch: {epoch + 1}/{epochs} - Loss: {epoch_loss:.4f}")

        return history

    def predict(self, user_vector):
        if len(user_vector.shape) == 1:
            user_vector = user_vector.unsqueeze(0)

        ph, _ = self.sample_h(user_vector)
        probabilities, _ = self.sample_v(ph)

        return probabilities

    def score(self, train_data, test_data):
        losses = []

        for i in range(train_data.shape[0]):
            user_vector = train_data[i : i + 1]
            vt = test_data[i : i + 1]

            if len(vt[vt >= 0]) == 0:
                continue

            pred = self.predict(user_vector)
            loss = torch.mean(torch.abs(vt[vt >= 0] - pred[vt >= 0]))
            losses.append(loss.item())

        return np.mean(losses)

## Training the RBM

In [11]:
nv = training_set.shape[1]
rbm = RBM(nv = nv, nh = 100, k = 10)

In [12]:
history = rbm.fit(
    training_set,
    epochs = 20,
    batch_size = 100
)

Epoch: 1/20 - Loss: 0.3298
Epoch: 2/20 - Loss: 0.2489
Epoch: 3/20 - Loss: 0.2502
Epoch: 4/20 - Loss: 0.2481
Epoch: 5/20 - Loss: 0.2468
Epoch: 6/20 - Loss: 0.2494
Epoch: 7/20 - Loss: 0.2468
Epoch: 8/20 - Loss: 0.2457
Epoch: 9/20 - Loss: 0.2475
Epoch: 10/20 - Loss: 0.2463
Epoch: 11/20 - Loss: 0.2446
Epoch: 12/20 - Loss: 0.2484
Epoch: 13/20 - Loss: 0.2450
Epoch: 14/20 - Loss: 0.2468
Epoch: 15/20 - Loss: 0.2438
Epoch: 16/20 - Loss: 0.2512
Epoch: 17/20 - Loss: 0.2453
Epoch: 18/20 - Loss: 0.2443
Epoch: 19/20 - Loss: 0.2492
Epoch: 20/20 - Loss: 0.2448


## Testing the RBM

In [13]:
test_loss = rbm.score(training_set, test_set)
print("Test Loss:", test_loss.round(6))

Test Loss: 0.244296


## Predicting for one user

In [15]:
user_id = 5
prediction = rbm.predict(training_set[user_id])
print(prediction)

tensor([[1.0000, 0.9728, 0.9568,  ..., 0.0509, 0.9615, 0.0563]])


## Recommending Movies

In [16]:
user_id = 5
pred = rbm.predict(training_set[user_id]).detach().numpy().flatten()

unseen = training_set[user_id].numpy() == -1
scores = pred.copy()
scores[~unseen] = -1

recommended = np.argsort(scores)[::-1][:10]
print(recommended)

[ 24  90  86  81  96  69 186 194 207 731]


## Printing the name of movies

In [17]:
for movie in recommended:
    print(f"{movie:<5}: {movies.iloc[movie, 1]}")

24   : Leaving Las Vegas (1995)
90   : Mary Reilly (1996)
86   : Dunston Checks In (1996)
81   : Antonia's Line (Antonia) (1995)
96   : Shopping (1994)
69   : From Dusk Till Dawn (1996)
186  : Prophecy, The (1995)
194  : Species (1995)
207  : White Man's Burden (1995)
731  : Ghost in the Shell (Kokaku kidotai) (1995)
